In [1]:
import torch, torch.nn as nn
import torchvision
from torchvision import datasets, transforms
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt
import numpy as np
from torch.nn.modules.activation import ReLU
from tqdm.auto import tqdm
from torch.utils.data import DataLoader

ModuleNotFoundError: No module named 'torchvision'

In [ ]:
device = 'cuda' if  torch.cuda.is_available() else 'cpu'

In [ ]:
#Training data
train_data = datasets.FashionMNIST(
    root = 'data',
    train = True,
    download = True,
    transform = ToTensor(),
    target_transform = None
)

test_data = datasets.FashionMNIST(
    root = 'data',
    train = False,
    download = True,
    transform = ToTensor(),
    target_transform = None

In [ ]:
data_loader_train = DataLoader(dataset = train_data,
                               batch_size = 32,
                               shuffle = True)
data_loader_test = DataLoader(dataset = test_data,
                              batch_size = 32,
                              shuffle = False)

In [ ]:
loss = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params = model.parameters(),
                            lr = 0.1)

In [ ]:
def eval_model(model: nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss: torch.nn.Module):
  now_loss =  0

  model.eval()
  with torch.inference_mode():
    for X, y in tqdm(data_loader):
      y_pred = model(X)
      now_loss += loss(y_pred, y)

    now_loss /= len(data_loader)

  return {"model name": model.__class__.__name__,
          "model loss": now_loss.item()}

model_0_results = eval_model(model = model,
                             data_loader = data_loader_test,
                             loss =  loss)

In [ ]:
def train_step(model: nn.Module,
                data_loader: torch.utils.data.DataLoader,
                loss: torch.nn.Module,
                optimizer: torch.optim.Optimizer,
                device:  torch.device = device):

  model.train()

  train_loss = 0
  for batch, (X, y) in enumerate(data_loader):
      X, y = X.to(device), y.to(device)
      y_pred = model.forward(X)
      loss_train = loss(y_pred, y)
      train_loss += loss_train

      optimizer.zero_grad()
      loss_train.backward()
      optimizer.step()

  train_loss /= len(data_loader)
  print(f"The average loss over epoch {epoch + 1} is {train_loss}")

In [ ]:
def test_step(model: nn.Module,
              data_loader: torch.utils.data.DataLoader,
              loss: torch.nn.Module,
              device: torch.device = device):
  test_loss = 0
  model.eval()
  with torch.inference_mode():
      for X, y in (data_loader_test):
        X, y = X.to(device), y.to(device)
        test_pred  = model(X)
        loss_test = loss(test_pred, y)
        test_loss += loss_test
      test_loss /= len(data_loader_test)
  print(f"The average loss over test_set is {test_loss}")

In [ ]:
from torch.nn.modules.pooling import MaxPool2d
class ConvModel(nn.Module):
  def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
    super().__init__()
    #Block 1 Start
    self.conv_1 = nn.Conv2d(in_channels = input_shape,
                           out_channels = hidden_units,
                           kernel_size = (2,2),
                           stride = 1,
                           padding = 1)
    self.relu_1 = nn.ReLU(),

    self.conv_2 = nn.Conv2d(in_channels = hidden_units,
                           out_channels = hidden_units,
                           kernel_size = 1,
                           stride = 1,
                           padding = 1)
    self.relu_2 = nn.ReLU(),
    self.maxpool_1 = nn.MaxPool2d(kernel_size = (2,2)),
    #Block 1 End

    #Block 2 Start
    self.conv_1 = nn.Conv2d(in_channels = hidden_units,
                           out_channels = hidden_units,
                           kernel_size = (2,2),
                           stride = 1,
                           padding = 1)
    self.relu_1 = nn.ReLU(),

    self.conv_2 = nn.Conv2d(in_channels = hidden_units,
                           out_channels = hidden_units,
                           kernel_size = 1,
                           stride = 1,
                           padding = 1)
    self.relu_2 = nn.ReLU(),
    self.maxpool_1 = nn.MaxPool2d(kernel_size = (2,2)),
    #Block 2 End
      
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features = hidden_units * 8* 8,
                  out_features = output_shape)
    )

  def forward(self, X):
    X =  X.to(device)
    X = self.block_1(X)
    print(X.shape)
    X = self.block_2(X)
    print(X.shape)
    X = self.classifier(X)
    return X

In [ ]:
epochs = 5

for epoch in tqdm(range(epochs)):
  print(f"Epoch {epoch+1}-------------\n")
  train_step(model = model_1,
             data_loader= data_loader_train,
             loss = loss,
             optimizer = optimizer,
             device = device)
  test_step(model = model_1,
            data_loader = data_loader_test,
            loss = loss,
            device = device)